In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pandas import DataFrame, read_excel, read_csv
from sklearn.preprocessing import MinMaxScaler
from sklearn.ensemble import GradientBoostingClassifier
import plotly.graph_objects as go

In [ ]:
filename = "../DataBase/DataCollection.xlsx"
dataset: DataFrame = read_excel(filename, sheet_name="791 Records")
boiling_data: DataFrame = read_excel(filename, sheet_name="991 Records")
systems = read_csv("./791Database_244Systems.txt", sep = ";")

#dataset_3methylnonane: DataFrame = read_excel(filename, sheet_name="3-methylnonane")
#system_solvent_screening = read_csv("./SolventScreeningSystems.txt", sep = ";")

boiling_dict = (boiling_data.drop_duplicates(subset="Solvent").set_index("Solvent")["Solvent boiling (K)"].to_dict())
dataset["Solvent boiling (K)"] = dataset["Solvent"].map(boiling_dict)


`Variables Enconding`

In [ ]:
dissolve_encoding: dict[str, int] = {"NO": 0,"YES": 1}
sample_type_enconding: dict[str, int] = {"pellet": 0, "waste": 1, "fiber": 2, "film": 3, "powder": 4}

encoding: dict[str, dict[str, int]] = {
    "Dissolution": dissolve_encoding,
    "Sample type": sample_type_enconding
}
df: DataFrame = dataset.replace(encoding, inplace=False)

In [ ]:
dataset_time = df["Time (min)"].dropna().to_numpy(dtype=int)
dataset_temperature = df["Temperature (K)"].dropna().to_numpy()

`Variables Normalization`

In [ ]:
df = df.drop(columns=["Polymer_ID", "Solvent_ID", "Polymer", "Solvent", "Solvent boiling (K)"])

X = df.drop(columns=["Dissolution"])
y = df["Dissolution"]

X_log = np.log1p(X)
min_max_scaler = MinMaxScaler(feature_range=(0, 1), copy=True)
X_scaled = min_max_scaler.fit_transform(X_log)

df_log_minmax = DataFrame(X_scaled, columns = X.columns, index= X.index)
df_log_minmax["Dissolution"] = y

In [ ]:
target = 'Dissolution'
X = df_log_minmax.drop(columns=target)
y = df_log_minmax[target]

`Prediction GB Models`

In [ ]:
# Hyperparameters
random_states = [198637118, 2275613469, 650646565, 2176125524, 1587810184]

param_grid = {
    'learning_rate': 0.15,
    'loss': 'log_loss',
    'max_depth': 3,
    'max_features': 0.5,
    'min_samples_leaf': 10,
    'min_samples_split': 20,
    'n_estimators': 35,
    'subsample': 0.9,
}

models = []
for i, rs in enumerate(random_states):
    random_state_model = rs
    model = GradientBoostingClassifier(**param_grid, random_state=random_state_model)
    model.fit(X, y)
    models.append(model)

In [ ]:
def get_unique_values(data, min_val, max_val):
    return np.sort(np.unique(data[(data >= min_val) & (data <= max_val)]))

In [ ]:
unique_time_values = get_unique_values(dataset_time, 1, 500)

In [ ]:
import os
os.makedirs("Surfaces", exist_ok=True)

In [ ]:
# Melting or degradation temperature (K) of each polymer
polymer_temperatures = {
    "ABS": 633.15,
    "PC": 553.15,
    "PS": 633.15,
    "LDPE": 383.15,
    "HDPE": 403.15,
    "PP": 438.15,
    "PVC": 473.15,
    "PET": 524.15,
    "PVDF": 483.15,
    "PU": 372.15,
    "PA6": 543.15,
    "PA66": 553.15
}

def evaluate_system(polymer_name, solvent_name, sample_type, dataset, min_max_scaler, encoding, polymer_temperatures):

    # Obtain polymer-solvent target system
    df_system = dataset[(dataset["Polymer"] == polymer_name) &(dataset["Solvent"] == solvent_name) &(dataset["Sample type"] == sample_type)].copy()
    df_system["Source"] = "Database"

    if df_system.empty:
        print(f"  No match for {polymer_name} / {solvent_name} / {sample_type}")
        print("   Polymer matches:", (dataset["Polymer"] == polymer_name).sum())
        print("   Solvent matches:", (dataset["Solvent"] == solvent_name).sum())
        print("   Sample type matches:", (dataset["Sample type"] == sample_type).sum())
        return None

    solvent_boiling = df_system["Solvent boiling (K)"].iloc[0]
    solvent_melting = df_system["Solvent melting (K)"].iloc[0]
    polymer_melting = polymer_temperatures.get(polymer_name)

    time_vals = np.unique(np.round(np.quantile(unique_time_values, np.linspace(0,1,100))).astype(int))

    temp_lower = max(dataset_temperature.min(), solvent_melting)
    temp_higher = min(polymer_melting, solvent_boiling)
    #temp_higher = min(383.15, solvent_boiling) LDPE Melting Constraint
    filtered_dataset_temperature = get_unique_values(dataset_temperature, min_val = temp_lower, max_val = temp_higher)
    temperature_vals = np.unique(np.round(np.quantile(filtered_dataset_temperature, np.linspace(0, 1, 50)), 2))

    prediction_only = df_system[["Time (min)", "Temperature (K)", "Dissolution"]].isna().all().all()

    row_template = df_system.iloc[0].copy()
    grid_records = []
    for t in time_vals:
        for temp in temperature_vals:
            row = row_template.copy()
            row["Time (min)"] = t
            row["Temperature (K)"] = temp
            grid_records.append(row)

    df_grid = pd.DataFrame(grid_records)
    df_grid["Source"] = "GridSearch"

    df_merged = df_grid.copy() if prediction_only else pd.concat([df_system, df_grid], ignore_index=True)

    drop_cols = ["Polymer_ID","Solvent_ID","Polymer","Solvent","Source","Solvent boiling (K)"]
    df_merged_encoded = (df_merged.replace(encoding, inplace=False).drop(columns=[c for c in drop_cols if c in df_merged.columns]))
    df_unnormalized = df_merged_encoded[["Time (min)", "Temperature (K)"]].rename(columns={"Time (min)": "Time (min) Unnormalized", "Temperature (K)": "Temperature (K) Unnormalized"})

    X = df_merged_encoded.drop(columns=["Dissolution"])
    y = df_merged_encoded["Dissolution"]
    X_log = np.log1p(X)
    X_scaled = min_max_scaler.transform(X_log)
    df_log_minmax = pd.DataFrame(X_scaled, columns=X.columns, index=X.index)
    df_log_minmax["Dissolution"] = y

    df_predict = pd.concat([df_log_minmax, df_unnormalized, df_merged[["Source"]]], axis=1)
    X_predict = df_predict.drop(columns=["Dissolution", "Time (min) Unnormalized", "Temperature (K) Unnormalized", "Source"])
    df_scored = df_predict.copy()

    for i, model in enumerate(models, 1):
      df_scored[f"Dissolution_{i}"] = model.predict_proba(X_predict)[:, 1]

    prob_cols = [f"Dissolution_{i}" for i in range(1, len(models) + 1)]
    df_scored["Dissolution_mean"] = df_scored[prob_cols].mean(axis=1)
    df_scored["Dissolution_std"] = df_scored[prob_cols].std(axis=1)

    database_mask = df_scored["Source"] == "Database"
    grid_mask = df_scored["Source"] == "GridSearch"

    df_scored.loc[database_mask, "Dissolution_std"] = 0.0
    df_scored.loc[grid_mask, "Dissolution"] = df_scored.loc[grid_mask, "Dissolution_mean"]

    fig = go.Figure()
    grid_subset = df_scored[df_scored["Source"] == "GridSearch"]
    if not grid_subset.empty:
        xs = np.sort(grid_subset['Time (min) Unnormalized'].unique())
        ys = np.sort(grid_subset['Temperature (K) Unnormalized'].unique())

        Z_mean = grid_subset.pivot_table(
            index='Temperature (K) Unnormalized',
            columns='Time (min) Unnormalized',
            values='Dissolution_mean'
        ).reindex(index=ys, columns=xs).values

        Z_std = grid_subset.pivot_table(
            index='Temperature (K) Unnormalized',
            columns='Time (min) Unnormalized',
            values='Dissolution_std'
        ).reindex(index=ys, columns=xs).values

        std_min = 0
        std_max = 0.130
        #std_max_solvent_screening = 0.210

        fig.add_trace(go.Surface(
            x=xs,
            y=ys,
            z=Z_mean,
            surfacecolor=Z_std,
            colorscale="RDYlBu",
            cmin=std_min,
            cmax=std_max,
            opacity=1,
            hovertemplate=(
                "Time (min): %{x:.1f}<br>"
                "Temperature (K): %{y:.2f}<br>"
                "Dissolution Probability (Mean): %{z:.3f}<extra></extra>"
            ),
            colorbar=dict(
                title=dict(
                    text="Std Dev",
                    side="top",  
                    font=dict(size=13),
                ),
                thickness=12,
                len=0.6,
                x=0.92,
                y=0.5,
                tickformat=".3f",
                dtick=0.010
            )
        ))

    df_valid_originals = df_scored[
        (df_scored["Source"] == "Database") & (~df_scored["Dissolution"].isna())
    ]
    for outcome, base_color, marker in [
        (0, "rgba(255,150,150,1)", "circle"),
        (1, "rgba(150,255,150,1)", "diamond")
    ]:
        subset = df_valid_originals[df_valid_originals["Dissolution"] == outcome]
        if subset.empty:
            continue
        label = "Dissolution" if outcome == 1 else "No Dissolution"
        fig.add_trace(go.Scatter3d(
            x=subset['Time (min) Unnormalized'],
            y=subset['Temperature (K) Unnormalized'],
            z=[outcome]*len(subset),
            mode="markers",
            marker=dict(size=7, color=base_color, symbol=marker, line=dict(color="black", width=1)),
            name=f"Experimental {label} Point",
            showlegend=True
        ))

    fig.update_layout(
        scene=dict(
            xaxis=dict(
                title=dict(text="Time (min)", font=dict(size=14)),
                tickfont=dict(size=12),
                range = [0, 500],
                dtick = 100,
            ),
            yaxis=dict(
                title=dict(text="Temperature (K)", font=dict(size=14)),
                tickfont=dict(size=12),
                range=[temp_lower, temp_higher + (temp_higher - temp_lower) * 0.15],
            ),
            zaxis=dict(
                title=dict(text="Dissolution Probability (Mean)", font=dict(size=14)),
                range=[-0.02, 1.0],
                tick0=0,
                dtick=0.10,
                tickformat=".3f",
                tickfont=dict(size=12)
            )
        ),
        title=dict(
            text=f"Dissolution of {polymer_name} ({sample_type}) in {solvent_name}",
            y=0.95,
            x=0.5,
            xanchor="center",
            yanchor="top"
        ),
        legend=dict(
            orientation="h",
            yanchor="middle",
            y=0.8,
            xanchor="left",
            x=0.45,
            bgcolor="rgba(255,255,255,0.75)",
            bordercolor="lightgray",
            borderwidth=1,
            itemsizing="trace"
        ),
        margin=dict(l=80, r=80, b=80, t=80, pad=10),
        autosize=True,
        height=900,
        width=1100,
        scene_aspectmode="cube"
    )

    save_name = f"{polymer_name}_{solvent_name}_{sample_type}".replace(" ", "_").replace("/", "-")
    fig.write_html(f"Surfaces/{save_name}.html")

In [ ]:
total = len(systems)
#total = len(system_solvent_screening)
#for i, row in system_solvent_screening.iterrows():
for i, row in systems.iterrows():
    print(f" Processing {i+1}/{total}: {row['Polymer']} - {row['Solvent']} - {row['Sample type']}")
    evaluate_system(row["Polymer"], row["Solvent"], row["Sample type"], dataset, min_max_scaler, encoding,polymer_temperatures)
    #evaluate_system(row["Polymer"], row["Solvent"], row["Sample type"], dataset_3methylnonane, min_max_scaler, encoding,polymer_temperatures)